In [1]:
import matplotlib
matplotlib.use('Agg')
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
import warnings; warnings.filterwarnings('ignore')
print('[harness] headless setup done')


[harness] headless setup done


In [2]:
import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from itertools import combinations
from xgboost import XGBClassifier
from sklearn.model_selection import BaseCrossValidator, ParameterGrid
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, roc_curve, log_loss
)

In [3]:
# ============================================
# Paths & factor pairs
# ============================================
# Alternate machine path (SARB laptop):
# BASE_DIR = Path('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/legacy')
BASE_DIR = Path('C:/Users/ccwis/OneDrive/Documents/1. MEng - Data Science/Project/factor_timing/legacy')

DATA_DIR = BASE_DIR / 'data_input'
OUTPUT_DIR = BASE_DIR / 'results' / 'model_outputs'
PROBS_DIR = OUTPUT_DIR / 'probs'
ARTIFACT_DIR = OUTPUT_DIR / 'artifacts'
SHAP_DIR = OUTPUT_DIR / 'shap'
METRICS_PATH = OUTPUT_DIR / 'metrics_summary.csv'

for _dir in (PROBS_DIR, ARTIFACT_DIR, SHAP_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# Every factor pair this pipeline runs for a given model.
FACTOR_PAIRS = ['Momentum_vs_Value', 'Momentum_vs_Quality', 'Value_vs_Quality']

In [4]:
# ============================================
# Data loading (parameterized by factor pair)
# ============================================
def load_data(pair):
    df_factor = pd.read_csv(DATA_DIR / 'labeled_relative_factor_returns.csv', parse_dates=['Date'])
    df_factor = df_factor[['Date', f'{pair}_trend']]
    df_factor = df_factor.set_index('Date')

    df = pd.read_csv(DATA_DIR / 'lagged_data.csv', parse_dates=['Date'])
    df = df.set_index('Date')

    data = pd.concat([df, df_factor], axis=1, join='inner')
    # Trend scanning needs returns out to t+10, so the final ~10 dates have
    # no computable label. ffill() would manufacture them from the last known
    # label, putting fabricated targets inside the OOS block -- drop instead.
    data = data.dropna(subset=[f'{pair}_trend'])
    # Unified missing-value policy across all three models.
    return data.ffill().bfill()

In [5]:
# ========================
# Phase 1: Combinatorial Purged CV
# ========================

CV_SPLITS = 5
CV_TEST_SPLITS = 1
PURGE_GAP = 12
FINAL_TEST_POINTS = 156   # 3 years of unseen data


class CombinatorialPurgedCV(BaseCrossValidator):
    """Purged Combinatorial Cross-Validation (De Prado style)."""

    def __init__(self, n_splits=CV_SPLITS, n_test_splits=CV_TEST_SPLITS, purge_gap=PURGE_GAP):
        self.n_splits = n_splits
        self.n_test_splits = n_test_splits
        self.purge_gap = purge_gap

    def split(self, X, y=None, groups=None):
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        indices = np.arange(n_samples)

        # Create fold boundaries
        fold_boundaries = [(i * fold_size, (i + 1) * fold_size) for i in range(self.n_splits)]
        fold_boundaries[-1] = (fold_boundaries[-1][0], n_samples)

        # Generate test fold combinations
        for test_fold_indices in combinations(range(self.n_splits), self.n_test_splits):
            test_mask = np.zeros(n_samples, dtype=bool)

            # Collect test indices
            for fold_idx in test_fold_indices:
                start, end = fold_boundaries[fold_idx]
                test_mask[start:end] = True

            # Train mask starts as complement
            train_mask = ~test_mask

            # Purge around each selected test fold
            for fold_idx in test_fold_indices:
                start, end = fold_boundaries[fold_idx]

                # purge before
                train_mask[max(0, start - self.purge_gap): start] = False
                # purge after
                train_mask[end: min(n_samples, end + self.purge_gap)] = False

            yield indices[train_mask], indices[test_mask]

    def get_n_splits(self, X=None, y=None, groups=None):
        from math import comb
        return comb(self.n_splits, self.n_test_splits)

In [6]:
def window_signature(X):
    """
    Create a stable, compact signature for a rolling window
    based on its mean and std across features.
    Used for hyperparameter caching.
    """
    import hashlib
    stats = np.concatenate([X.mean(axis=0), X.std(axis=0)])
    stats_bytes = stats.tobytes()
    return hashlib.sha1(stats_bytes).hexdigest()

In [7]:
def tune_hyperparameters(X, y, return_grid=False, verbose=True):
    """Grid search over CPCV folds."""
    cv = CombinatorialPurgedCV()

    param_grid = {
        'learning_rate': [0.01, 0.1, 0.5],     # slightly more realistic grid
        'max_depth': [3, 6, 10],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'base_score': [0.5, 0.6, 0.7]
    }

    best_score, best_params = -np.inf, None
    rows = []

    if verbose:
        print(f"Testing {len(ParameterGrid(param_grid))} parameter combinations...")

    for params in ParameterGrid(param_grid):

        fold_scores = []

        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model = XGBClassifier(
                objective='binary:logistic',
                eval_metric='logloss',
                n_estimators=500,
                n_jobs=1,        # subsample's RNG stream varies with
                random_state=0,  # thread count -- pin both to reproduce
                **params
            )

            model.fit(X_train, y_train, verbose=False)

            # IMPORTANT: use probabilities, not class labels
            y_prob = model.predict_proba(X_test)[:, 1]
            score = roc_auc_score(y_test, y_prob)
            fold_scores.append(float(score))

        mean_auc = np.mean(fold_scores)

        rows.append({
            **params,
            "mean_auc": mean_auc,
            "std_auc": np.std(fold_scores, ddof=1),
            "min_auc": np.min(fold_scores),
            "max_auc": np.max(fold_scores),
            "fold_aucs": fold_scores
        })

        if mean_auc > best_score:
            best_score, best_params = mean_auc, params
            if verbose:
                print(f"New best params: {best_params} (AUC: {best_score:.3f})")

    grid_df = pd.DataFrame(rows).sort_values("mean_auc", ascending=False)

    if verbose:
        print("\n=== Top 5 Parameter Combinations ===")
        for _, r in grid_df.head(5).iterrows():
            row_params = {k: r[k] for k in param_grid}
            print(f"Score: {r['mean_auc']:.3f} | Params: {row_params}")

    if return_grid:
        return best_params, grid_df
    return best_params

In [8]:
def save_artifact(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    elif path.suffix == ".json":
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)
    else:
        with open(path, "wb") as f:
            pickle.dump(obj, f)

In [9]:
def walk_forward_validation(
    X, y, n_test, base_params,
    retune_every=52, lookback=260
):
    """
    Rolling-window walk-forward with adaptive hyperparameter tuning
    and hyperparameter caching based on window signatures.
    """

    actuals, predictions, probabilities = [], [], []

    # IMPORTANT
    current_params = base_params.copy()

    hyperparam_cache = {}   # signature -> best_params (dict)

    print(f"\nRunning Rolling-Window Walk-Forward Validation...")
    print(f"Lookback window = {lookback} samples")

    for i in tqdm(range(n_test)):
        train_size = len(X) - n_test + i
        if train_size <= 0:
            continue

        # ===== Purge gap =====
        # Trend-scanning labels (span=(1,12)) resolve using returns out to
        # t+10, so training rows within PURGE_GAP of the test point have
        # labels computed from the test period itself. Drop them.
        train_end = train_size - PURGE_GAP
        if train_end <= 0:
            continue

        # Rolling window, measured from the purged end so the window
        # length stays at `lookback`.
        start = max(0, train_end - lookback)
        X_train = X[start:train_end]
        y_train = y[start:train_end]

        X_test  = X[train_size:train_size+1]
        y_test  = y[train_size:train_size+1]

        # ======================================================
        # Adaptive hyperparameter tuning + CACHE
        # ======================================================
        if retune_every is not None and i > 0 and i % retune_every == 0:

            sig = window_signature(X_train)

            if sig in hyperparam_cache:
                print(f"\nReusing cached params at step {i}")
                current_params = hyperparam_cache[sig].copy()

            else:
                print(f"\n--- Retuning at step {i}, train_size={train_size} ---")

                new_params = tune_hyperparameters(X_train, y_train)

                current_params = new_params.copy()
                hyperparam_cache[sig] = current_params.copy()


        # ======================================================
        # Fit model with current params
        # ======================================================

        model = XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            n_estimators=500,
            n_jobs=1,
            random_state=0,
            **current_params
        )

        model.fit(X_train, y_train, verbose=False)

        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        predictions.append(int(y_pred[0]))
        actuals.append(int(y_test[0]))
        probabilities.append(float(y_prob[0]))


    print("\n=== Final Evaluation Results ===")
    print(classification_report(actuals, predictions))

    return (
        np.array(actuals),
        np.array(predictions),
        np.array(probabilities)
    )

In [10]:
# ========================
# Phase 3: Plotting
# ========================
def plot_wfv_results(actuals, predictions, probabilities, pair_label=""):

    # ---- classification metrics
    acc  = accuracy_score(actuals, predictions)
    f1   = f1_score(actuals, predictions)
    rec  = recall_score(actuals, predictions)
    prec = precision_score(actuals, predictions)

    print(f"Walk-forward Performance [{pair_label}]:")
    print(f"Accuracy : {acc:.3f}")
    print(f"F1 Score : {f1:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"Precision: {prec:.3f}")

    # ---- probabilistic metrics
    auc    = roc_auc_score(actuals, probabilities)
    pr_auc = average_precision_score(actuals, probabilities)
    ll     = log_loss(actuals, probabilities)

    print(f"ROC AUC :   {auc:.4f}")
    print(f"PR  AUC :   {pr_auc:.4f}")
    print(f"Log Loss:   {ll:.4f}")

    # ---- timeline
    plt.figure(figsize=(12, 6))
    plt.plot(actuals, label='Actual', marker='o',linestyle='None')
    plt.plot(predictions, label='Predicted', marker='x',linestyle='None')
    plt.plot(probabilities, label='Probability', linestyle='--', alpha=0.5)
    plt.title(f'Walk-Forward Validation Results [{pair_label}]')
    plt.legend()
    plt.show()

    # ---- ROC curve
    fpr, tpr, _ = roc_curve(actuals, probabilities)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f'ROC AUC = {auc:.3f}')
    plt.plot([0,1],[0,1],'--k')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve [{pair_label}]")
    plt.legend()
    plt.show()

    return {
        "accuracy": acc, "f1": f1, "recall": rec, "precision": prec,
        "roc_auc": auc, "pr_auc": pr_auc, "log_loss": ll,
    }

In [11]:
# ============================================
# Metrics summary — single source of truth for the write-up.
# Re-running a pair updates its row instead of duplicating it.
# ============================================
def append_metrics_row(row):
    row = {"timestamp": datetime.now(timezone.utc).isoformat(), **row}
    new_row = pd.DataFrame([row])
    if METRICS_PATH.exists():
        existing = pd.read_csv(METRICS_PATH)
        existing = existing[~(
            (existing["model"] == row["model"]) & (existing["factor_pair"] == row["factor_pair"])
        )]
        combined = pd.concat([existing, new_row], ignore_index=True)
    else:
        combined = new_row
    combined.to_csv(METRICS_PATH, index=False)

In [12]:
# ============================================
# XGBoost SHAP feature importance (Tree SHAP)
# ============================================
def compute_shap_importance(data, best_params, feature_names, n_test):
    import shap

    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    X = np.nan_to_num(X, nan=np.nan, posinf=1e10, neginf=-1e10)

    train_end = len(X) - n_test
    X_train_final = X[:train_end]
    y_train_final = y[:train_end]

    final_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_jobs=1,
        random_state=0,
        **best_params
    )
    final_model.fit(X_train_final, y_train_final)

    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_train_final)

    importance = pd.Series(
        np.abs(shap_values).mean(axis=0),
        index=feature_names
    ).sort_values(ascending=False)

    return importance, shap_values, X_train_final

In [13]:
# ============================================
# Full pipeline for one factor pair
# ============================================
def run_pipeline(pair):
    pair_slug = pair.lower()
    print(f"\n{'='*60}\nFACTOR PAIR: {pair}\n{'='*60}")

    data = load_data(pair)

    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    feature_names = data.columns[:-1].tolist()

    # Handle inf/NaN (XGBoost handles NaNs natively, but we cap inf)
    X = np.nan_to_num(X, nan=np.nan, posinf=1e10, neginf=-1e10)

    print("=== PHASE 1: INITIAL HYPERPARAMETER TUNING ===")
    # Tune only on pre-test data:
    X_train_full = X[:-FINAL_TEST_POINTS]
    y_train_full = y[:-FINAL_TEST_POINTS]

    best_params, tuning_grid = tune_hyperparameters(X_train_full, y_train_full, return_grid=True)
    save_artifact(tuning_grid, ARTIFACT_DIR / f"xgboost_{pair_slug}_tuning_grid.pkl")
    save_artifact(best_params, ARTIFACT_DIR / f"xgboost_{pair_slug}_best_params.json")

    print("\n=== PHASE 2: ADAPTIVE ROLLING WALK-FORWARD ===")
    actuals, preds, probs = walk_forward_validation(
        X, y,
        n_test=FINAL_TEST_POINTS,
        base_params=best_params,
        retune_every=52,      # yearly retuning
        lookback=260          # 5-year rolling window (260 weekly bars)
    )

    print("\n=== ADAPTIVE PARAMS RESULTS ===")
    metrics = plot_wfv_results(actuals, preds, probs, pair_label=pair)

    df_probs = pd.DataFrame({
        "Date": data.index[-len(probs):],
        "actual": actuals,
        "prediction": preds,
        "probability": probs,
    }).set_index("Date")
    df_probs.to_csv(PROBS_DIR / f"xgboost_{pair_slug}_probs.csv", index=True)

    shap_importance, shap_values, X_train_final = compute_shap_importance(
        data, best_params, feature_names, FINAL_TEST_POINTS
    )
    shap_importance.head(20).to_csv(SHAP_DIR / f"xgboost_{pair_slug}_shap.csv", index=True)

    append_metrics_row({
        "model": "XGBoost",
        "factor_pair": pair,
        "n_test": FINAL_TEST_POINTS,
        **metrics,
        "best_params": json.dumps(best_params),
    })

    return {
        "pair": pair,
        "best_params": best_params,
        "tuning_grid": tuning_grid,
        "feature_names": feature_names,
        "adaptive": (actuals, preds, probs),
        "shap_importance": shap_importance,
        "shap_values": shap_values,
        "X_train_final": X_train_final,
    }

In [14]:
def plot_pair_diagnostics(result):
    import shap

    pair = result["pair"]

    plt.figure(figsize=(10, 12))
    result["shap_importance"].head(20).plot(kind='barh', color='purple')
    plt.title(f"XGBoost SHAP Global Feature Importance\n(Mean |SHAP|) [{pair}]")
    plt.xlabel("Mean(|SHAP|)")
    plt.gca().invert_yaxis()
    plt.grid(axis='x', linestyle='--', alpha=0.4)
    plt.show()

    shap.summary_plot(result["shap_values"], result["X_train_final"],
                      feature_names=result["feature_names"], plot_type="dot")
    shap.summary_plot(result["shap_values"], result["X_train_final"],
                      feature_names=result["feature_names"], plot_type="bar")

In [15]:
# ============================================
# Run every factor pair
# ============================================
all_results = {}
for pair in FACTOR_PAIRS:
    all_results[pair] = run_pipeline(pair)
    plot_pair_diagnostics(all_results[pair])

print("\nDone. Metrics summary written to:", METRICS_PATH)


FACTOR PAIR: Momentum_vs_Value
=== PHASE 1: INITIAL HYPERPARAMETER TUNING ===
Testing 243 parameter combinations...


New best params: {'base_score': 0.5, 'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 3, 'subsample': 0.6} (AUC: 0.554)


New best params: {'base_score': 0.5, 'colsample_bytree': 0.6, 'learning_rate': 0.01, 'max_depth': 6, 'subsample': 0.6} (AUC: 0.554)


New best params: {'base_score': 0.5, 'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 3, 'subsample': 0.6} (AUC: 0.568)


New best params: {'base_score': 0.5, 'colsample_bytree': 0.8, 'learning_rate': 0.5, 'max_depth': 6, 'subsample': 1.0} (AUC: 0.571)


New best params: {'base_score': 0.5, 'colsample_bytree': 0.8, 'learning_rate': 0.5, 'max_depth': 10, 'subsample': 0.6} (AUC: 0.579)


... (521 outputs left out)

100%|██████████| 156/156 [14:11<00:00,  1.56it/s]

100%|██████████| 156/156 [14:11<00:00,  5.46s/it]


=== Final Evaluation Results ===
              precision    recall  f1-score   support

           0       0.55      0.39      0.46        82
           1       0.49      0.65      0.56        74

    accuracy                           0.51       156
   macro avg       0.52      0.52      0.51       156
weighted avg       0.52      0.51      0.51       156


=== ADAPTIVE PARAMS RESULTS ===
Walk-forward Performance [Value_vs_Quality]:
Accuracy : 0.513
F1 Score : 0.558
Recall   : 0.649
Precision: 0.490
ROC AUC :   0.5160
PR  AUC :   0.4984
Log Loss:   1.3102



Done. Metrics summary written to: C:\Users\ccwis\OneDrive\Documents\1. MEng - Data Science\Project\factor_timing\legacy\results\model_outputs\metrics_summary.csv
